# Transfer Learning BO with MultitaskGP

Two MultitaskGP surrogates:
1. **Burr regression** — predicts burr height (continuous), shared across source + target tasks via ICM kernel
2. **Feasibility classification** — predicts feasibility label (0/1), also shared via ICM kernel

The BO loop uses **Constrained Expected Improvement**: EI from the burr model, weighted by P(feasible) from the feasibility model.

In [202]:
import pickle
import numpy as np
import torch
import pandas as pd
import plotly.graph_objects as go
from botorch.models import MultiTaskGP
from botorch.models.transforms.outcome import Standardize
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition import ExpectedImprovement
from botorch.optim import optimize_acqf
from gpytorch.mlls import ExactMarginalLogLikelihood

torch.set_default_dtype(torch.float64)
SEED = 42
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

## 1. Load data & pick source / target tasks

In [203]:
with open("./data/task_df_dict.pkl", "rb") as f:
    task_df_dict = pickle.load(f)

FEATURE_COLS = ["feedrate", "gas_pressure", "focal_position"]

# Choose the target task
TARGET_TASK = "150_ST150MD0-N2H0-30-2_L76_0.4_10000_H450"

# Source tasks: all others that have both feasible and infeasible points
source_tasks = []
for name, df in task_df_dict.items():
    if name == TARGET_TASK:
        continue
    has_neg = ((df["burr_evaluated"] < 0) | (df["roughness_z_evaluated"] < 0)).any()
    has_pos = ((df["burr_evaluated"] >= 0) & (df["roughness_z_evaluated"] >= 0)).any()
    if has_neg and has_pos:
        source_tasks.append(name)

print(f"Target task: {TARGET_TASK} ({len(task_df_dict[TARGET_TASK])} points)")
print(f"Source tasks: {len(source_tasks)}")
for s in source_tasks:
    print(f"  {s}: {len(task_df_dict[s])} points")

Target task: 150_ST150MD0-N2H0-30-2_L76_0.4_10000_H450 (99 points)
Source tasks: 23
  100_ST100MD0-N2H0-30-2_L76_0.4_10000_S235JR: 140 points
  100_ST100MD0-N2H0-30-2_L76_0.4_10000_16Mo3: 52 points
  100_ST100MD0-N2H0-30-2_L81_0.7_10000_S235JR: 175 points
  100_ST100MD0-N2H0-30-2_L95_0.7_10000_S355JR: 52 points
  150_ST150MD0-N2H0-30-2_L76_0.4_10000_S235JR: 111 points
  150_ST150MD0-N2H0-30-2_L76_0.4_10000_16Mo3: 79 points
  150_ST150MD0-N2H0-30-2_L76_0.4_10000_S355J2: 52 points
  150_ST150MD0-N2H0-30-2_L76_0.4_10000_H400: 89 points
  150_ST150MD0-N2H0-30-2_L76_0.4_10000_S355J2sand: 74 points
  30_ST030MD4-N2S0-30-2_L26_0.5_6000_S235JR: 28 points
  50_ST050MD4-N2S0-30-2_L26_0.5_6000_S235JR: 170 points
  50_ST050MD5-N2S0-30-2_L76_4.0_10000_S235JR: 130 points
  60_ST060MD0-N2H0-30-2_L76_0.6_10000_16Mo3: 65 points
  60_ST060MD0-N2H0-30-2_L76_0.6_10000_DST400: 65 points
  60_ST060MD0-N2H0-30-2_L76_0.6_10000_S355J2: 65 points
  60_ST060MD0-N2H0-30-2_L76_0.7_6000_16Mo3: 53 points
  80_ST080M

## 2. Build the simulator (ground-truth oracle for the target task)

In [204]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor

def create_task_simulator(task_name, task_df_dict):
    """Build a simulator that returns burr value (or -1 if infeasible)."""
    df = task_df_dict[task_name]
    X = df[FEATURE_COLS].values
    burr = df["burr_evaluated"].values
    roughness = df["roughness_z_evaluated"].values

    # Feasibility label: 1 if both burr >= 0 and roughness >= 0
    y_feas = ((burr >= 0) & (roughness >= 0)).astype(int)

    # Classifier
    scaler_clf = StandardScaler().fit(X)
    clf = SVC(kernel="rbf", gamma="scale", probability=True, random_state=42)
    clf.fit(scaler_clf.transform(X), y_feas)

    # Regressor on feasible points only
    feas_mask = y_feas == 1
    scaler_reg = StandardScaler().fit(X[feas_mask])
    reg = GradientBoostingRegressor(n_estimators=200, random_state=42)
    reg.fit(scaler_reg.transform(X[feas_mask]), burr[feas_mask])

    def simulator(feedrate, gas_pressure, focal_position):
        x = np.atleast_2d([feedrate, gas_pressure, focal_position])
        is_feas = clf.predict(scaler_clf.transform(x))
        if is_feas[0] == 0:
            return -1.0
        return float(reg.predict(scaler_reg.transform(x))[0])

    return simulator

simulator = create_task_simulator(TARGET_TASK, task_df_dict)
print("Simulator ready.")

Simulator ready.


## 3. Prepare source-task data tensors

In [205]:
def make_feasibility_label(df):
    """1 if feasible (both burr & roughness >= 0), else 0."""
    return ((df["burr_evaluated"] >= 0) & (df["roughness_z_evaluated"] >= 0)).astype(float).values

# Assign integer task IDs: 0 = target, 1..N = source tasks
task_id_map = {TARGET_TASK: 0}
for i, name in enumerate(source_tasks):
    task_id_map[name] = i + 1

# --- Per-task normalisation ---
# Features: each task's X is scaled to [0, 1] using its own min/max — different
# tasks operate in different parameter ranges, so normalising per-task aligns
# them in the MTGP input space.
# Burr: standardised (mean=0, std=1) using each task's feasible points only —
# absolute burr scales differ across tasks, so standardising lets the ICM
# kernel focus on shape/correlation rather than scale.
task_feature_bounds = {}   # name -> (lo, hi)  shape (d,)
task_burr_stats     = {}   # name -> (mean, std)

for name, df in task_df_dict.items():
    X = df[FEATURE_COLS].values
    lo = torch.tensor(X.min(axis=0), dtype=torch.float64)
    hi = torch.tensor(X.max(axis=0), dtype=torch.float64)
    task_feature_bounds[name] = (lo, hi)

    burr = df["burr_evaluated"].values
    feas = burr[burr >= 0]
    mu = float(feas.mean()) if feas.size > 0 else 0.0
    sd = float(feas.std())  if feas.size > 1 else 1.0
    task_burr_stats[name] = (mu, sd if sd > 0 else 1.0)

def normalise_X(X_raw: torch.Tensor, name: str) -> torch.Tensor:
    lo, hi = task_feature_bounds[name]
    return (X_raw - lo) / (hi - lo).clamp_min(1e-12)

def standardise_burr(y_raw: torch.Tensor, name: str) -> torch.Tensor:
    mu, sd = task_burr_stats[name]
    return (y_raw - mu) / sd

# Build source data for both models
source_X_burr_list, source_Y_burr_list = [], []
source_X_feas_list, source_Y_feas_list = [], []

for name in source_tasks:
    df = task_df_dict[name]
    tid = task_id_map[name]
    X_raw = torch.tensor(df[FEATURE_COLS].values, dtype=torch.float64)
    X_norm = normalise_X(X_raw, name)
    task_col = torch.full((len(X_raw), 1), tid, dtype=torch.float64)
    X_with_task = torch.cat([X_norm, task_col], dim=-1)  # (n, d+1)

    # Feasibility model: all points, label in {0, 1}
    y_feas = torch.tensor(make_feasibility_label(df), dtype=torch.float64).unsqueeze(-1)
    source_X_feas_list.append(X_with_task)
    source_Y_feas_list.append(y_feas)

    # Burr model: feasible points only, standardised per-task
    feas_mask = df["burr_evaluated"].values >= 0
    if feas_mask.sum() > 0:
        y_burr_raw = torch.tensor(df["burr_evaluated"].values[feas_mask], dtype=torch.float64)
        y_burr = standardise_burr(y_burr_raw, name).unsqueeze(-1)
        source_X_burr_list.append(X_with_task[feas_mask])
        source_Y_burr_list.append(y_burr)

source_X_burr = torch.cat(source_X_burr_list, dim=0)
source_Y_burr = torch.cat(source_Y_burr_list, dim=0)
source_X_feas = torch.cat(source_X_feas_list, dim=0)
source_Y_feas = torch.cat(source_Y_feas_list, dim=0)

print(f"Source burr data:        {source_X_burr.shape[0]} points  "
      f"(features in [0,1], burr standardised per task)")
print(f"Source feasibility data: {source_X_feas.shape[0]} points  (features in [0,1])")
print(f"Burr Y range across all tasks: [{source_Y_burr.min():.2f}, {source_Y_burr.max():.2f}]  (z-units)")

Source burr data:        1889 points  (features in [0,1], burr standardised per task)
Source feasibility data: 2270 points  (features in [0,1])
Burr Y range across all tasks: [-1.91, 5.30]  (z-units)


## 4. Compute bounds for the target task

In [206]:
df_target = task_df_dict[TARGET_TASK]

# Original-space bounds: used to call the simulator and to un-normalise candidates.
target_lo, target_hi = task_feature_bounds[TARGET_TASK]
target_bounds = torch.stack([target_lo, target_hi])  # (2, d)

# Model-space bounds: per-task normalisation puts each task on the unit cube,
# so the acquisition function is optimised over [0, 1]^d.
target_bounds_norm = torch.stack([
    torch.zeros(len(FEATURE_COLS), dtype=torch.float64),
    torch.ones(len(FEATURE_COLS),  dtype=torch.float64),
])

target_burr_mu, target_burr_sd = task_burr_stats[TARGET_TASK]

print("Target bounds (original space):")
for i, c in enumerate(FEATURE_COLS):
    print(f"  {c}: [{target_bounds[0, i]:.2f}, {target_bounds[1, i]:.2f}]")
print(f"\nTarget burr standardisation: mean={target_burr_mu:.3f}, std={target_burr_sd:.3f}")
print("Model input bounds: unit cube [0, 1]^d  (per-task normalised)")

Target bounds (original space):
  feedrate: [1.50, 2.55]
  gas_pressure: [1.50, 4.50]
  focal_position: [-12.10, -7.10]

Target burr standardisation: mean=1318.441, std=261.527
Model input bounds: unit cube [0, 1]^d  (per-task normalised)


---

# Didactic: Optimal Transport for Transfer-Learning Classification

**The idea in one sentence.** If a *source* task and a *target* task live in the same feature space but have shifted input distributions, we can learn a coupling `γ` between source and target samples that minimises the total transport cost, **move the source points to the target support**, and then train a classifier on the transported source — which now "looks like" target data while keeping the source labels.

**Pipeline**:
1. Pick one source task + the target task (same features, shared binary label: feasible / infeasible).
2. Compute the entropic-regularised OT plan `γ` between source features and target features (Sinkhorn).
3. Barycentric-map source samples onto target: `x_s_transported = γ · X_t / γ.sum(axis=1)`.
4. Train a classifier on the transported source; evaluate on held-out target points.
5. Compare to the no-transfer baseline (train on target labels only) and on raw source (no OT).

In [207]:
# --- POT may need to be installed ---
# !pip install POT
import numpy as np
import ot
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Reuse the dicts defined earlier in the notebook:
# task_df_dict, TARGET_TASK, source_tasks, FEATURE_COLS, task_feature_bounds

def feas_label(df):
    return ((df["burr_evaluated"] >= 0) & (df["roughness_z_evaluated"] >= 0)).astype(int).values

# --- Pick ONE source task (largest available) for the didactic demo ---
SOURCE_TASK = max(source_tasks, key=lambda k: len(task_df_dict[k]))
print(f"Source task for demo: {SOURCE_TASK}  ({len(task_df_dict[SOURCE_TASK])} points)")
print(f"Target task:          {TARGET_TASK}  ({len(task_df_dict[TARGET_TASK])} points)")

# --- Build source and target feature/label arrays ---
df_s = task_df_dict[SOURCE_TASK]
df_t = task_df_dict[TARGET_TASK]

Xs_raw = df_s[FEATURE_COLS].values
ys = feas_label(df_s)
Xt_raw = df_t[FEATURE_COLS].values
yt = feas_label(df_t)

# --- Small labeled target pool + target test set ---
Xt_labeled, Xt_test, yt_labeled, yt_test = train_test_split(
    Xt_raw, yt, train_size=15, stratify=yt, random_state=0
)

# --- Per-task [0, 1] normalisation (consistent with Section 3) ---
# Each task is scaled to the unit cube using its OWN min/max bounds (computed
# in cell-7's `task_feature_bounds`). This puts source and target in a common
# reference frame so the OT cost matrix is meaningful, without leaking target
# scale info from the tiny 15-point labeled subset.
def to_unit_cube(X_raw, task_name):
    lo, hi = task_feature_bounds[task_name]
    lo_np, hi_np = lo.numpy(), hi.numpy()
    return (X_raw - lo_np) / np.clip(hi_np - lo_np, 1e-12, None)

Xs     = to_unit_cube(Xs_raw,     SOURCE_TASK)
Xt_lab = to_unit_cube(Xt_labeled, TARGET_TASK)
Xt_te  = to_unit_cube(Xt_test,    TARGET_TASK)

print(f"Source (unit-cube):         {Xs.shape}, feasible rate = {ys.mean():.2f}")
print(f"Target labeled (unit-cube): {Xt_lab.shape}, feasible rate = {yt_labeled.mean():.2f}")
print(f"Target test (unit-cube):    {Xt_te.shape}")

Source task for demo: 80_ST080MD5-N2S0-30-2_L76_4.0_10000_S355  (268 points)
Target task:          150_ST150MD0-N2H0-30-2_L76_0.4_10000_H450  (99 points)
Source (unit-cube):         (268, 3), feasible rate = 0.91
Target labeled (unit-cube): (15, 3), feasible rate = 0.73
Target test (unit-cube):    (84, 3)


## Step 1 — Visualise source and target distributions (before any transport)

Even though features are 3-D (`feedrate`, `gas_pressure`, `focal_position`), we project onto
the first two for 2-D visuals. The **shift between the two clouds is exactly what OT compensates for.**

In [208]:
import plotly.graph_objects as go

def scatter_xy(X, y, name, symbol, color_feas, color_infeas, size=10):
    f = y == 1
    out = []
    if f.any():
        out.append(go.Scatter(
            x=X[f, 0], y=X[f, 1], mode="markers", name=f"{name} (feasible)",
            marker=dict(size=size, color=color_feas, symbol=symbol, line=dict(width=1, color="black")),
        ))
    if (~f).any():
        out.append(go.Scatter(
            x=X[~f, 0], y=X[~f, 1], mode="markers", name=f"{name} (infeasible)",
            marker=dict(size=size, color=color_infeas, symbol=symbol, line=dict(width=1, color="black")),
        ))
    return out

fig = go.Figure()
for tr in scatter_xy(Xs,     ys,          "source", "circle", "seagreen", "indianred", size=8):
    fig.add_trace(tr)
for tr in scatter_xy(Xt_lab, yt_labeled,  "target labeled", "diamond", "limegreen", "crimson", size=13):
    fig.add_trace(tr)
fig.update_layout(
    title="Source vs Target distributions (2 of 3 features)<br>"
          "<sup>Each task scaled to its own [0, 1] cube — any remaining offset is real distribution shift.</sup>",
    xaxis_title=f"{FEATURE_COLS[0]} (per-task [0, 1])",
    yaxis_title=f"{FEATURE_COLS[1]} (per-task [0, 1])",
    width=800, height=550,
)
fig.show()

## Step 2 — Compute the OT plan γ (Sinkhorn)

The **cost matrix** `M[i, j] = ||x_s_i − x_t_j||²` measures how "expensive" it is to map source-i to target-j. Sinkhorn finds the plan `γ` that:

- Minimises `⟨γ, M⟩` (total weighted cost) …
- … subject to marginals `γ·1 = a` (source mass) and `γᵀ·1 = b` (target mass), …
- … with entropic regularisation `−reg · H(γ)` to make the problem smooth and fast.

The result `γ[i, j]` is the amount of mass transported from source-i to target-j.

In [209]:
# Uniform marginals: each source point has mass 1/n_s, each target point has mass 1/n_t
a = np.ones(len(Xs))    / len(Xs)
b = np.ones(len(Xt_lab)) / len(Xt_lab)

# Squared-Euclidean cost
M = ot.dist(Xs, Xt_lab, metric="sqeuclidean")
M /= M.max()                       # normalise for numerical stability

reg = 0.05
gamma = ot.sinkhorn(a, b, M, reg=reg, numItermax=200)

print(f"OT plan shape: {gamma.shape}  (n_source x n_target_labeled)")
print(f"Total transported mass: {gamma.sum():.4f}  (should be ~1.0)")
print(f"Max single entry:       {gamma.max():.4f}")
print(f"Avg transport cost:     {(gamma * M).sum():.4f}")

OT plan shape: (268, 15)  (n_source x n_target_labeled)
Total transported mass: 1.0000  (should be ~1.0)
Max single entry:       0.0029
Avg transport cost:     0.1063


## Step 3 — Visualise the transport plan

Each line connects a source point to a target point whose coupling `γ[i, j]` is non-negligible.
**Line opacity is proportional to transported mass.** The plan effectively shows which target
points each source point is being "pulled towards".

In [210]:
# Keep only the strongest connections (top 2% of non-zero mass) for readability
thresh = np.quantile(gamma[gamma > 0], 0.98)
idx_s, idx_t = np.where(gamma >= thresh)
weights = gamma[idx_s, idx_t]
weights_norm = weights / weights.max()

fig = go.Figure()

# Source and target clouds
for tr in scatter_xy(Xs,     ys,          "source", "circle",  "seagreen",  "indianred", size=8):
    fig.add_trace(tr)
for tr in scatter_xy(Xt_lab, yt_labeled,  "target",  "diamond", "limegreen", "crimson",  size=13):
    fig.add_trace(tr)

# Transport lines
for k in range(len(idx_s)):
    i, j = idx_s[k], idx_t[k]
    fig.add_trace(go.Scatter(
        x=[Xs[i, 0], Xt_lab[j, 0]],
        y=[Xs[i, 1], Xt_lab[j, 1]],
        mode="lines",
        line=dict(color=f"rgba(100,100,250,{0.1 + 0.6*weights_norm[k]:.2f})", width=1),
        showlegend=False, hoverinfo="skip",
    ))

fig.update_layout(
    title=f"Sinkhorn OT plan γ  (reg = {reg})<br><sup>Line opacity ∝ transported mass. Top 2% of couplings shown.</sup>",
    xaxis_title=f"{FEATURE_COLS[0]} (per-task [0, 1])",
    yaxis_title=f"{FEATURE_COLS[1]} (per-task [0, 1])",
    width=850, height=600,
)
fig.show()

## Step 4 — Barycentric map: transport the source onto the target

Given γ, the **barycentric projection** of a source point `x_s_i` is the γ-weighted mean of target points:

$$ \hat{x}_{s,i} \;=\; \frac{\sum_j \gamma_{ij}\, x_{t,j}}{\sum_j \gamma_{ij}} $$

Intuitively: each source point is moved to the centroid of the target points it was strongly coupled to.
**The labels stay with the source** — we only move the features.

In [211]:
# Barycentric mapping
row_sums = gamma.sum(axis=1, keepdims=True)
Xs_transported = (gamma @ Xt_lab) / np.clip(row_sums, 1e-12, None)

fig = go.Figure()
# Original source (faded)
for tr in scatter_xy(Xs, ys, "source (original)", "circle", "lightgreen", "lightcoral", size=6):
    tr.marker.opacity = 0.35
    fig.add_trace(tr)
# Transported source
for tr in scatter_xy(Xs_transported, ys, "source (transported)", "circle", "seagreen", "indianred", size=9):
    fig.add_trace(tr)
# Target reference
for tr in scatter_xy(Xt_lab, yt_labeled, "target labeled", "diamond", "limegreen", "crimson", size=13):
    fig.add_trace(tr)

# Arrows from original source to transported source
for i in range(len(Xs)):
    fig.add_trace(go.Scatter(
        x=[Xs[i, 0], Xs_transported[i, 0]],
        y=[Xs[i, 1], Xs_transported[i, 1]],
        mode="lines",
        line=dict(color="rgba(80,80,80,0.25)", width=1),
        showlegend=False, hoverinfo="skip",
    ))

fig.update_layout(
    title="Barycentric transport: original source → transported source<br>"
          "<sup>Transported source should overlap the target cloud while preserving source labels.</sup>",
    xaxis_title=f"{FEATURE_COLS[0]} (per-task [0, 1])",
    yaxis_title=f"{FEATURE_COLS[1]} (per-task [0, 1])",
    width=850, height=600,
)
fig.show()

## Step 5 — Classify: does OT transfer actually help?

Compare three classifiers on the **target test set**:

| Model | Trained on |
|---|---|
| (a) target-only baseline | the 15 labelled target points |
| (b) source-only (no OT) | raw source features + source labels |
| (c) **OT-transfer** | **transported** source features + source labels |

In [212]:
# (a) Target-only baseline
clf_tgt = DecisionTreeClassifier(max_depth=5, random_state=0).fit(Xt_lab, yt_labeled)
acc_tgt = accuracy_score(yt_test, clf_tgt.predict(Xt_te))

# (b) Source-only baseline (no transport)
clf_src = DecisionTreeClassifier(max_depth=5, random_state=0).fit(Xs, ys)
acc_src = accuracy_score(yt_test, clf_src.predict(Xt_te))

# (c) OT-transfer: classifier trained on transported source
clf_ot = DecisionTreeClassifier(max_depth=5, random_state=0).fit(Xs_transported, ys)
acc_ot = accuracy_score(yt_test, clf_ot.predict(Xt_te))

print(f"{'Method':<30} {'Target-test accuracy':>22}")
print("-" * 54)
print(f"{'(a) target-only (15 labels)':<30} {acc_tgt:>22.3f}")
print(f"{'(b) source-only (no OT)':<30} {acc_src:>22.3f}")
print(f"{'(c) OT-transfer (Sinkhorn)':<30} {acc_ot:>22.3f}")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=["target-only", "source-only", "OT-transfer"],
    y=[acc_tgt, acc_src, acc_ot],
    marker_color=["steelblue", "orange", "seagreen"],
    text=[f"{v:.3f}" for v in [acc_tgt, acc_src, acc_ot]],
    textposition="outside",
))
fig.update_layout(
    title="Target-test Accuracy — effect of Optimal Transport",
    yaxis_title="Accuracy", yaxis=dict(range=[0, 1.05]),
    width=700, height=450,
)
fig.show()

Method                           Target-test accuracy
------------------------------------------------------
(a) target-only (15 labels)                     0.821
(b) source-only (no OT)                         0.786
(c) OT-transfer (Sinkhorn)                      0.738


### Takeaways

- **OT is unsupervised in the target labels** — Step 2 only used target *features* (`Xt_lab` coordinates), never `yt_labeled`. Target labels were only used later to train/evaluate classifiers.
- **The plan γ is the key object.** Once you have it, you can either (i) barycentric-map the source as shown here, or (ii) plug it into a more elaborate loss (JDOT couples γ with a label cost, DeepJDOT makes the encoder end-to-end trainable).
- **Failure mode — negative transfer.** If the source and target have *different class-conditional distributions* (e.g. the OT mass crosses class boundaries), transport will drag source-positives onto target-negatives and the classifier will degrade. Class-regularised variants like `SinkhornL1l2Transport` add a group-sparsity penalty on γ that prevents this.
- **This is the feasibility building-block** used inside `SinkhornL1l2Transport` / `JCPOTTransport` in the companion `transfer_learning_adapt.ipynb` notebook — the machinery is the same, just with extra label-aware regularisation.

---

## Step 6 — Mitigating negative transfer

Vanilla Sinkhorn dropped accuracy because (1) the OT plan is class-blind and (2) barycentric projection
collapses 268 source points onto only 15 target centroids. Two principled fixes:

- **(a) Class-regularised OT — `SinkhornL1l2Transport`.** Adds a group-lasso penalty on γ that
  suppresses couplings crossing the feasibility boundary. Uses `ys` and `yt_labeled` as side info
  (so it's *semi-supervised* rather than fully unsupervised in the labels).
- **(c) `MappingTransport`.** Instead of `Xs_transported = γ @ Xt_lab` (a γ-weighted average), it
  fits a smooth mapping `f: source → target` (Gaussian kernel). The transported source spreads
  across target space rather than collapsing onto centroids, *and* `f` generalises to unseen source
  points — useful when you actually want to deploy the transport later.

In [213]:
# (a) Class-regularised OT — group-lasso penalty on γ that suppresses
# couplings crossing class boundaries. Needs source labels (`ys`) and the
# small target labelled set (`yt_labeled`) so it knows which class each point
# belongs to. `reg_e` is the entropic term; `reg_cl` is the class penalty.
from ot.da import SinkhornL1l2Transport

ot_l1l2 = SinkhornL1l2Transport(reg_e=0.05, reg_cl=1.0, max_iter=20)
ot_l1l2.fit(Xs=Xs, ys=ys, Xt=Xt_lab, yt=yt_labeled)
Xs_l1l2 = ot_l1l2.transform(Xs=Xs)

fig = go.Figure()
for tr in scatter_xy(Xs,      ys,         "source (original)",     "circle",  "lightgreen", "lightcoral", size=6):
    tr.marker.opacity = 0.3
    fig.add_trace(tr)
for tr in scatter_xy(Xs_l1l2, ys,         "source (class-reg OT)", "circle",  "seagreen",   "indianred",  size=9):
    fig.add_trace(tr)
for tr in scatter_xy(Xt_lab,  yt_labeled, "target labeled",        "diamond", "limegreen",  "crimson",    size=13):
    fig.add_trace(tr)

fig.update_layout(
    title="Class-regularised OT (SinkhornL1l2Transport)<br>"
          "<sup>Group-lasso on γ keeps source-feasible mass with target-feasible mass.</sup>",
    xaxis_title=f"{FEATURE_COLS[0]} (per-task [0, 1])",
    yaxis_title=f"{FEATURE_COLS[1]} (per-task [0, 1])",
    width=850, height=600,
)
fig.show()

In [214]:
# (c) MappingTransport — learn a smooth f(x_s) ≈ barycentric_target.
# Spreads source over target space (no centroid collapse) and lets you
# transport unseen source points later.
from ot.da import MappingTransport

ot_map = MappingTransport(
    kernel="gaussian", eta=1e-5, mu=1.0, sigma=1.0, bias=True, max_iter=20,
)
ot_map.fit(Xs=Xs, Xt=Xt_lab)
Xs_map = ot_map.transform(Xs=Xs)

fig = go.Figure()
for tr in scatter_xy(Xs,     ys,         "source (original)",      "circle",  "lightgreen", "lightcoral", size=6):
    tr.marker.opacity = 0.3
    fig.add_trace(tr)
for tr in scatter_xy(Xs_map, ys,         "source (kernel mapped)", "circle",  "seagreen",   "indianred",  size=9):
    fig.add_trace(tr)
for tr in scatter_xy(Xt_lab, yt_labeled, "target labeled",         "diamond", "limegreen",  "crimson",    size=13):
    fig.add_trace(tr)

fig.update_layout(
    title="MappingTransport (Gaussian-kernel mapping)<br>"
          "<sup>Smooth f(x_s) avoids barycentric collapse onto 15 target centroids.</sup>",
    xaxis_title=f"{FEATURE_COLS[0]} (per-task [0, 1])",
    yaxis_title=f"{FEATURE_COLS[1]} (per-task [0, 1])",
    width=850, height=600,
)
fig.show()

In [215]:
# Compare every variant on the same target test set
def _acc(X_tr, y_tr):
    clf = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_tr, y_tr)
    return accuracy_score(yt_test, clf.predict(Xt_te))

results = {
    "target-only (15 labels)":   _acc(Xt_lab,         yt_labeled),
    "source-only (no OT)":       _acc(Xs,             ys),
    "OT barycentric (Sinkhorn)": _acc(Xs_transported, ys),
    "class-reg OT (L1l2)":       _acc(Xs_l1l2,        ys),
    "MappingTransport":          _acc(Xs_map,         ys),
}

print(f"{'Method':<28} {'Target-test accuracy':>22}")
print("-" * 52)
for k, v in results.items():
    print(f"{k:<28} {v:>22.3f}")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(results.keys()),
    y=list(results.values()),
    marker_color=["steelblue", "orange", "tomato", "seagreen", "mediumpurple"],
    text=[f"{v:.3f}" for v in results.values()],
    textposition="outside",
))
fig.update_layout(
    title="Target-test accuracy: vanilla OT vs class-regularised vs mapping",
    yaxis_title="Accuracy", yaxis=dict(range=[0, 1.05]),
    width=900, height=500,
)
fig.show()

Method                         Target-test accuracy
----------------------------------------------------
target-only (15 labels)                       0.821
source-only (no OT)                           0.786
OT barycentric (Sinkhorn)                     0.738
class-reg OT (L1l2)                           0.702
MappingTransport                              0.702


---

## Step 7 — Rotate the target across all 24 tasks

A single (source, target) pair tells you very little about whether OT is useful in general — the
result we just saw could be a lucky or unlucky neighbour. Here we **treat every task in
`task_df_dict` as the target in turn**, pair it with the largest *other* task as source, and rerun
the same five classifiers. Then we plot the distribution of accuracies per method so we can ask:

- Does any OT variant beat plain `source-only` *on average*?
- How often does each method **win** per target?
- Are there target tasks where transfer is uniformly hopeless (suggests the task is genuinely an outlier)?

Setup is identical to Steps 1–6: per-task `[0, 1]` features, 15-point stratified labelled set,
Decision Tree (`max_depth=5`) classifier on each variant, evaluated on the held-out target test split.

In [216]:
# --- Run the OT analysis once per task, rotating the target ---
import warnings
from contextlib import redirect_stdout, redirect_stderr
import io

def run_ot_analysis(target_name, train_size=15, sinkhorn_reg=0.05, random_state=0):
    """Full OT pipeline for one target task. Returns dict of accuracies (NaN on failure)."""
    df_t = task_df_dict[target_name]
    yt_full = feas_label(df_t)

    # Need ≥2 of each class for stratified split with ≥1 in test
    if (yt_full == 1).sum() < 2 or (yt_full == 0).sum() < 2:
        return None

    # Source: largest OTHER task that has both classes
    candidates = [
        n for n in task_df_dict
        if n != target_name
        and (feas_label(task_df_dict[n]) == 1).sum() > 0
        and (feas_label(task_df_dict[n]) == 0).sum() > 0
    ]
    if not candidates:
        return None
    src = max(candidates, key=lambda k: len(task_df_dict[k]))

    df_s = task_df_dict[src]
    Xs_raw = df_s[FEATURE_COLS].values
    ys_loc = feas_label(df_s)
    Xt_raw = df_t[FEATURE_COLS].values

    n_lab = min(train_size, len(yt_full) - 5)
    if n_lab < 4:
        return None
    try:
        Xt_lab_raw, Xt_te_raw, yt_lab, yt_te = train_test_split(
            Xt_raw, yt_full, train_size=n_lab,
            stratify=yt_full, random_state=random_state,
        )
    except ValueError:
        return None

    # Per-task [0, 1] normalisation (Section 3 scheme)
    Xs_n     = to_unit_cube(Xs_raw,     src)
    Xt_lab_n = to_unit_cube(Xt_lab_raw, target_name)
    Xt_te_n  = to_unit_cube(Xt_te_raw,  target_name)

    # 1) Vanilla Sinkhorn + barycentric projection
    a = np.ones(len(Xs_n)) / len(Xs_n)
    b = np.ones(len(Xt_lab_n)) / len(Xt_lab_n)
    M = ot.dist(Xs_n, Xt_lab_n, metric="sqeuclidean")
    M = M / max(M.max(), 1e-12)
    g = ot.sinkhorn(a, b, M, reg=sinkhorn_reg, numItermax=200)
    Xs_bary = (g @ Xt_lab_n) / np.clip(g.sum(axis=1, keepdims=True), 1e-12, None)

    # 2) Class-regularised OT (silence convergence warnings + chatty stdout)
    Xs_l1l2 = None
    try:
        with warnings.catch_warnings(), redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            warnings.simplefilter("ignore")
            ot_l1l2 = SinkhornL1l2Transport(reg_e=sinkhorn_reg, reg_cl=1.0, max_iter=20)
            ot_l1l2.fit(Xs=Xs_n, ys=ys_loc, Xt=Xt_lab_n, yt=yt_lab)
            Xs_l1l2 = ot_l1l2.transform(Xs=Xs_n)
    except Exception:
        pass

    # 3) MappingTransport
    Xs_map = None
    try:
        with warnings.catch_warnings(), redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            warnings.simplefilter("ignore")
            ot_m = MappingTransport(
                kernel="gaussian", eta=1e-5, mu=1.0, sigma=1.0, bias=True, max_iter=20,
            )
            ot_m.fit(Xs=Xs_n, Xt=Xt_lab_n)
            Xs_map = ot_m.transform(Xs=Xs_n)
    except Exception:
        pass

    def acc(X_tr, y_tr):
        if X_tr is None: return np.nan
        clf = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_tr, y_tr)
        return accuracy_score(yt_te, clf.predict(Xt_te_n))

    return {
        "source_task":      src,
        "n_target":         len(yt_full),
        "target_feas_pct":  float(yt_full.mean()),
        "target-only":      acc(Xt_lab_n, yt_lab),
        "source-only":      acc(Xs_n, ys_loc),
        "OT barycentric":   acc(Xs_bary, ys_loc),
        "class-reg OT":     acc(Xs_l1l2, ys_loc),
        "MappingTransport": acc(Xs_map, ys_loc),
    }

rows, skipped = [], []
for name in task_df_dict:
    res = run_ot_analysis(name)
    if res is None:
        skipped.append(name)
        continue
    rows.append({"target_task": name, **res})

results_df = pd.DataFrame(rows)
print(f"Completed: {len(results_df)} / {len(task_df_dict)} tasks   (skipped: {len(skipped)})")
if skipped:
    print("Skipped:", skipped)
results_df.head()

Completed: 23 / 27 tasks   (skipped: 4)
Skipped: ['30_ST030MD4-N2S0-30-2_L26_0.5_6000_S235JR', '60_ST060MD0-O2S0-30-2_L76_0.8_10000_S235JR', '80_ST080MD0-N2H0-30-2_L26_0.4_12000_S235', '80_ST100MD0-N2H0-30-2_L26_0.4_12000_S235']


,target_task,source_task,n_target,target_feas_pct,target-only,source-only,OT barycentric,class-reg OT,MappingTransport
0,100_ST100MD0-N2H0-30-2_L76_0.4_10000_S235JR,80_ST080MD5-N2S0-30-2_L76_4.0_10000_S355,140,0.878571,0.824000,0.848000,0.768000,0.808000,0.912000
1,100_ST100MD0-N2H0-30-2_L76_0.4_10000_16Mo3,80_ST080MD5-N2S0-30-2_L76_4.0_10000_S355,52,0.807692,0.756757,0.756757,0.756757,0.756757,0.810811
2,100_ST100MD0-N2H0-30-2_L81_0.7_10000_S235JR,80_ST080MD5-N2S0-30-2_L76_4.0_10000_S355,175,0.840000,0.843750,0.918750,0.918750,0.893750,0.850000
3,100_ST100MD0-N2H0-30-2_L95_0.7_10000_S355JR,80_ST080MD5-N2S0-30-2_L76_4.0_10000_S355,52,0.903846,0.891892,0.675676,0.486486,0.675676,0.702703
4,150_ST150MD0-N2H0-30-2_L76_0.4_10000_S235JR,80_ST080MD5-N2S0-30-2_L76_4.0_10000_S355,111,0.963964,0.916667,0.791667,0.791667,0.833333,0.895833


In [217]:
# --- Visualise the rotation results ---
methods = ["target-only", "source-only", "OT barycentric", "class-reg OT", "MappingTransport"]
colors  = ["steelblue", "orange", "tomato", "seagreen", "mediumpurple"]

# 1) Boxplot — distribution of accuracy per method across all targets
fig = go.Figure()
for m, c in zip(methods, colors):
    fig.add_trace(go.Box(
        y=results_df[m].dropna(),
        name=m, marker_color=c,
        boxpoints="all", jitter=0.3, pointpos=-1.6,
    ))
fig.update_layout(
    title=f"Target-test accuracy across {len(results_df)} target tasks",
    yaxis_title="Accuracy", yaxis=dict(range=[0, 1.05]),
    width=900, height=500,
)
fig.show()

# 2) Heatmap — winner per row bolded and outlined
short_names = [n if len(n) <= 36 else n[:33] + "..." for n in results_df["target_task"]]
heat = results_df[methods].values
heat_for_argmax = np.where(np.isnan(heat), -np.inf, heat)
best_col = np.argmax(heat_for_argmax, axis=1)

text_matrix = [
    [
        ("" if np.isnan(v)
         else (f"<b>{v:.2f}</b>" if j == best_col[i] else f"{v:.2f}"))
        for j, v in enumerate(row)
    ]
    for i, row in enumerate(heat)
]

fig = go.Figure(data=go.Heatmap(
    z=heat, x=methods, y=short_names,
    text=text_matrix,
    texttemplate="%{text}", textfont=dict(size=10),
    colorscale="RdYlGn", zmin=0, zmax=1,
    colorbar=dict(title="accuracy"),
))
# Outline the winning cell on each row
for i, j in enumerate(best_col):
    fig.add_shape(
        type="rect", xref="x", yref="y",
        x0=j - 0.5, x1=j + 0.5,
        y0=i - 0.5, y1=i + 0.5,
        line=dict(color="black", width=2.5),
        fillcolor="rgba(0,0,0,0)",
        layer="above",
    )
fig.update_layout(
    title=f"Accuracy heatmap ({len(results_df)} target tasks × {len(methods)} methods) — winner per row outlined",
    width=900, height=max(450, 24 * len(results_df)),
    yaxis=dict(autorange="reversed"),
)
fig.show()

# 3) Numerical summary — mean / median / win rate vs source-only baseline
summary = results_df[methods].agg(["mean", "median", "std"]).T.round(3)
wins_vs_src = (results_df[methods].sub(results_df["source-only"], axis=0) > 0).sum()
wins_vs_tgt = (results_df[methods].sub(results_df["target-only"], axis=0) > 0).sum()
summary["wins_vs_source-only"] = wins_vs_src
summary["wins_vs_target-only"] = wins_vs_tgt
summary["best_count"] = (results_df[methods].rank(axis=1, ascending=False, method="min") == 1).sum()
print("Per-method summary across all rotated targets:\n")
print(summary)

Per-method summary across all rotated targets:

                   mean  median    std  wins_vs_source-only  \
target-only       0.862   0.887  0.090                   12   
source-only       0.825   0.838  0.106                    0   
OT barycentric    0.772   0.768  0.112                    3   
class-reg OT      0.800   0.808  0.084                    6   
MappingTransport  0.790   0.812  0.102                    8   

                  wins_vs_target-only  best_count  
target-only                         0          13  
source-only                         9           8  
OT barycentric                      4           1  
class-reg OT                        6           3  
MappingTransport                    9           3  


---

## Step 8 — Improving transfer further

Putting recommendations 3, 4, 5, 6 from the discussion into one rotation:

- **(#3) Better classifier head** — `GradientBoostingClassifier` instead of `DecisionTreeClassifier(max_depth=5)`. Smoother decision surface on continuous transported features.
- **(#4) Source picking by similarity** — for each target, rank candidate sources by Sinkhorn OT cost between their feature clouds (cheap, parameter-free, label-free) and pick the *most similar* instead of the largest.
- **(#5) Multi-source OT** — pool the top-3 most-similar sources and run a single class-regularised OT against the full target X. Uses much more transferable data than a single neighbour.
- **(#6) Burr-augmented OT (JDOT flavour)** — concatenate per-task standardised burr to the feature vector before computing the OT cost. Encourages couplings that match in *both* feature space and continuous burr value, not just feature space.

Compared methods (all use GBM head and full target X for OT):

| Name | What it changes |
|---|---|
| `target_only` | GBM on the 15 labelled target points (#3 only) |
| `source_largest_only` | GBM on the largest other source — closest to old (b) baseline |
| `source_similar_only` | GBM on the most-similar source (#3 + #4 without OT) |
| `classreg_OT_similar` | `SinkhornL1l2Transport` from most-similar source (#3 + #4) |
| `multisource_OT` | `SinkhornL1l2Transport` on pooled top-3 similar (#3 + #4 + #5) |
| `multisource_OT_burr` | Same but with burr column appended to OT features (#3 + #4 + #5 + #6) |

In [218]:
# --- Helpers for the improved pipeline ---
from sklearn.ensemble import GradientBoostingClassifier

# (#3) Better classifier head — gradient-boosted trees on the (transported) source.
def make_clf():
    return GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=0)

# (#4) Source picking by Sinkhorn similarity instead of "largest".
def sinkhorn_cost(Xs, Xt, reg=0.05):
    """Entropic OT cost between two unit-cube point clouds (used as a similarity)."""
    a = np.ones(len(Xs)) / len(Xs)
    b = np.ones(len(Xt)) / len(Xt)
    M = ot.dist(Xs, Xt, metric="sqeuclidean")
    M = M / max(M.max(), 1e-12)
    g = ot.sinkhorn(a, b, M, reg=reg, numItermax=200)
    return float((g * M).sum())

def rank_sources_by_similarity(target_name, Xt_unit):
    """Return [(name, cost), ...] sorted ascending. Uses ALL target X (not just labelled)."""
    cands = [
        n for n in task_df_dict
        if n != target_name
        and (feas_label(task_df_dict[n]) == 1).sum() > 0
        and (feas_label(task_df_dict[n]) == 0).sum() > 0
    ]
    scored = []
    for src in cands:
        Xs_n = to_unit_cube(task_df_dict[src][FEATURE_COLS].values, src)
        scored.append((src, sinkhorn_cost(Xs_n, Xt_unit)))
    return sorted(scored, key=lambda kv: kv[1])

# (#6) JDOT flavour — concatenate per-task standardised burr to the feature vector
# so the OT cost matches feasibility-and-burr together. Infeasible points get a
# sentinel z-value so they cluster away from feasibles. `scale` trades off feature
# distance vs label distance: at 0.5, burr's squared contribution ~= one feature axis.
def augment_with_burr(X_unit, burr_raw, task_name, infeasible_z=-3.0, scale=0.5):
    mu, sd = task_burr_stats[task_name]
    z = np.where(burr_raw >= 0, (burr_raw - mu) / sd, infeasible_z)
    return np.concatenate([X_unit, scale * z[:, None]], axis=1)


def run_ot_analysis_v2(target_name, train_size=15, top_k=3, sinkhorn_reg=0.05, random_state=0):
    """Improved pipeline: GBM head, similarity-picked source, multi-source pool, burr-aug OT."""
    df_t = task_df_dict[target_name]
    yt_full = feas_label(df_t)
    if (yt_full == 1).sum() < 2 or (yt_full == 0).sum() < 2:
        return None

    Xt_all_n = to_unit_cube(df_t[FEATURE_COLS].values, target_name)
    burr_t   = df_t["burr_evaluated"].values

    n_lab = min(train_size, len(yt_full) - 5)
    if n_lab < 4:
        return None
    try:
        idx_lab, idx_te = train_test_split(
            np.arange(len(yt_full)), train_size=n_lab,
            stratify=yt_full, random_state=random_state,
        )
    except ValueError:
        return None
    Xt_lab_n = Xt_all_n[idx_lab]; yt_lab = yt_full[idx_lab]; bt_lab = burr_t[idx_lab]
    Xt_te_n  = Xt_all_n[idx_te];  yt_te  = yt_full[idx_te]

    # Rank candidate sources by Sinkhorn cost vs FULL target X (label-free)
    ranked = rank_sources_by_similarity(target_name, Xt_all_n)
    if not ranked:
        return None
    src_similar = ranked[0][0]
    src_largest = max((s for s, _ in ranked), key=lambda k: len(task_df_dict[k]))

    def src_data(name):
        df = task_df_dict[name]
        return (
            to_unit_cube(df[FEATURE_COLS].values, name),
            feas_label(df),
            df["burr_evaluated"].values,
        )

    def acc_clf(X_tr, y_tr):
        if X_tr is None or len(X_tr) < 2 or len(np.unique(y_tr)) < 2:
            return np.nan
        return accuracy_score(yt_te, make_clf().fit(X_tr, y_tr).predict(Xt_te_n))

    Xs_L, ys_L, _    = src_data(src_largest)
    Xs_S, ys_S, bs_S = src_data(src_similar)

    out = {
        "source_largest":      src_largest,
        "source_similar":      src_similar,
        "sim_cost_top1":       ranked[0][1],
        "target_only":         acc_clf(Xt_lab_n, yt_lab),
        "source_largest_only": acc_clf(Xs_L, ys_L),
        "source_similar_only": acc_clf(Xs_S, ys_S),
    }

    # (#3 + #4) class-reg OT, single best source, full target X for the OT plan
    try:
        with warnings.catch_warnings(), redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            warnings.simplefilter("ignore")
            ot_l1l2 = SinkhornL1l2Transport(reg_e=sinkhorn_reg, reg_cl=1.0, max_iter=20)
            ot_l1l2.fit(Xs=Xs_S, ys=ys_S, Xt=Xt_lab_n, yt=yt_lab)
            Xs_l1l2 = ot_l1l2.transform(Xs=Xs_S)
        out["classreg_OT_similar"] = acc_clf(Xs_l1l2, ys_S)
    except Exception:
        out["classreg_OT_similar"] = np.nan

    # (#5) multi-source: pool top-K most-similar sources, run a single class-reg OT
    top_sources = [s for s, _ in ranked[:top_k]]
    pooled      = [src_data(s) for s in top_sources]
    Xs_pool = np.concatenate([X for X, _, _ in pooled])
    ys_pool = np.concatenate([y for _, y, _ in pooled])
    try:
        with warnings.catch_warnings(), redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            warnings.simplefilter("ignore")
            ot_multi = SinkhornL1l2Transport(reg_e=sinkhorn_reg, reg_cl=1.0, max_iter=20)
            ot_multi.fit(Xs=Xs_pool, ys=ys_pool, Xt=Xt_lab_n, yt=yt_lab)
            Xs_multi = ot_multi.transform(Xs=Xs_pool)
        out["multisource_OT"] = acc_clf(Xs_multi, ys_pool)
    except Exception:
        out["multisource_OT"] = np.nan

    # (#6) multi-source + burr-augmented features (JDOT flavour)
    try:
        Xs_aug = np.concatenate([
            augment_with_burr(X_s, b_s, name)
            for (X_s, _, b_s), name in zip(pooled, top_sources)
        ])
        Xt_lab_aug = augment_with_burr(Xt_lab_n, bt_lab, target_name)
        with warnings.catch_warnings(), redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            warnings.simplefilter("ignore")
            ot_jdot = SinkhornL1l2Transport(reg_e=sinkhorn_reg, reg_cl=1.0, max_iter=20)
            ot_jdot.fit(Xs=Xs_aug, ys=ys_pool, Xt=Xt_lab_aug, yt=yt_lab)
            Xs_jdot_aug = ot_jdot.transform(Xs=Xs_aug)
        # Drop the burr column for the downstream classifier (target test has no burr label)
        out["multisource_OT_burr"] = acc_clf(Xs_jdot_aug[:, :-1], ys_pool)
    except Exception:
        out["multisource_OT_burr"] = np.nan

    return out

In [219]:
# --- Run the improved pipeline once per target ---
rows_v2, skipped_v2 = [], []
for name in task_df_dict:
    res = run_ot_analysis_v2(name)
    if res is None:
        skipped_v2.append(name)
        continue
    rows_v2.append({"target_task": name, **res})

results_df_v2 = pd.DataFrame(rows_v2)
print(f"Completed: {len(results_df_v2)} / {len(task_df_dict)} tasks   (skipped: {len(skipped_v2)})")

methods_v2 = [
    "target_only",
    "source_largest_only",
    "source_similar_only",
    "classreg_OT_similar",
    "multisource_OT",
    "multisource_OT_burr",
]
labels_v2 = [
    "target-only (GBM)",
    "source-only (largest, GBM)",
    "source-only (similar, GBM)",
    "class-reg OT (similar, GBM)",
    "multi-source OT (top-3 similar)",
    "multi-source OT + burr (JDOT)",
]
colors_v2 = ["steelblue", "orange", "gold", "seagreen", "mediumpurple", "darkmagenta"]

# 1) Boxplot
fig = go.Figure()
for m, lab, c in zip(methods_v2, labels_v2, colors_v2):
    fig.add_trace(go.Box(
        y=results_df_v2[m].dropna(), name=lab, marker_color=c,
        boxpoints="all", jitter=0.3, pointpos=-1.6,
    ))
fig.update_layout(
    title=f"Improved methods across {len(results_df_v2)} target tasks",
    yaxis_title="Accuracy", yaxis=dict(range=[0, 1.05]),
    width=950, height=500,
)
fig.show()

# 2) Heatmap — winner per row bolded and outlined
short_names = [n if len(n) <= 36 else n[:33] + "..." for n in results_df_v2["target_task"]]
heat = results_df_v2[methods_v2].values
heat_for_argmax = np.where(np.isnan(heat), -np.inf, heat)
best_col = np.argmax(heat_for_argmax, axis=1)

text_matrix = [
    [
        ("" if np.isnan(v)
         else (f"<b>{v:.2f}</b>" if j == best_col[i] else f"{v:.2f}"))
        for j, v in enumerate(row)
    ]
    for i, row in enumerate(heat)
]

fig = go.Figure(data=go.Heatmap(
    z=heat, x=labels_v2, y=short_names,
    text=text_matrix,
    texttemplate="%{text}", textfont=dict(size=10),
    colorscale="RdYlGn", zmin=0, zmax=1,
    colorbar=dict(title="accuracy"),
))
# Outline the winning cell on each row
for i, j in enumerate(best_col):
    fig.add_shape(
        type="rect", xref="x", yref="y",
        x0=j - 0.5, x1=j + 0.5,
        y0=i - 0.5, y1=i + 0.5,
        line=dict(color="black", width=2.5),
        fillcolor="rgba(0,0,0,0)",
        layer="above",
    )
fig.update_layout(
    title="Accuracy heatmap — improved methods (winner per row outlined)",
    width=950, height=max(450, 24 * len(results_df_v2)),
    yaxis=dict(autorange="reversed"),
)
fig.show()

# 3) Numerical summary
summary = results_df_v2[methods_v2].agg(["mean", "median", "std"]).T.round(3)
summary.index = labels_v2
summary["wins_vs_target_only"] = [
    (results_df_v2[m] > results_df_v2["target_only"]).sum() for m in methods_v2
]
summary["wins_vs_largest"] = [
    (results_df_v2[m] > results_df_v2["source_largest_only"]).sum() for m in methods_v2
]
summary["best_count"] = (
    results_df_v2[methods_v2].rank(axis=1, ascending=False, method="min") == 1
).sum().values
print("Per-method summary:\n")
print(summary)

Completed: 23 / 27 tasks   (skipped: 4)


Per-method summary:

                                  mean  median    std  wins_vs_target_only  \
target-only (GBM)                0.870   0.890  0.073                    0   
source-only (largest, GBM)       0.827   0.844  0.106                    9   
source-only (similar, GBM)       0.813   0.800  0.120                    8   
class-reg OT (similar, GBM)      0.761   0.766  0.112                    4   
multi-source OT (top-3 similar)  0.707   0.735  0.122                    3   
multi-source OT + burr (JDOT)    0.803   0.826  0.106                    6   

                                 wins_vs_largest  best_count  
target-only (GBM)                             13          11  
source-only (largest, GBM)                     0           5  
source-only (similar, GBM)                    12           8  
class-reg OT (similar, GBM)                    7           0  
multi-source OT (top-3 similar)                4           0  
multi-source OT + burr (JDOT)                 10      

---

## Step 9 — Active labelling instead of random 15 points

Step 8 spent its label budget randomly. With only 15 points we can do better:
train a classifier on the *most-similar source* (no target labels yet), use its
`predict_proba` on all target points, and pick the 15 *most uncertain* points to
label — i.e. the points nearest the source classifier's decision boundary.

Reason: random samples mostly land where the source classifier is already
confident, so labelling them is redundant. Boundary-uncertain points carry the
most information per label, so 15 active labels are typically worth 25–40 random
ones.

Everything else in the pipeline is identical to Step 8 — only the labelling
strategy changes, so the Δ heatmap below directly shows the labelling-strategy
effect.


In [220]:
# --- (#5) Active labelling: pick uncertain target points instead of random ---
def pick_active_labels_idx(Xt_unit, src_clf, n_labels=15):
    """Uncertainty-sample n_labels target indices using a source-pretrained classifier.
    Returns indices ordered by descending uncertainty (most-uncertain first)."""
    proba = src_clf.predict_proba(Xt_unit)
    if proba.shape[1] < 2:
        return np.arange(min(n_labels, len(Xt_unit)))
    pos = proba[:, 1]
    uncertainty = -np.abs(pos - 0.5)   # closer to 0.5 ⇒ higher (more uncertain)
    return np.argsort(-uncertainty)[:n_labels]


def run_ot_analysis_v3(target_name, train_size=15, top_k=3, sinkhorn_reg=0.05, random_state=0):
    """Same as v2 but the 15 target labels are picked by uncertainty sampling
    using a classifier pretrained on the most-similar source."""
    df_t = task_df_dict[target_name]
    yt_full = feas_label(df_t)
    if (yt_full == 1).sum() < 2 or (yt_full == 0).sum() < 2:
        return None

    Xt_all_n = to_unit_cube(df_t[FEATURE_COLS].values, target_name)
    burr_t   = df_t["burr_evaluated"].values
    n_lab = min(train_size, len(yt_full) - 5)
    if n_lab < 4:
        return None

    ranked = rank_sources_by_similarity(target_name, Xt_all_n)
    if not ranked:
        return None
    src_similar = ranked[0][0]
    src_largest = max((s for s, _ in ranked), key=lambda k: len(task_df_dict[k]))

    def src_data(name):
        df = task_df_dict[name]
        return (
            to_unit_cube(df[FEATURE_COLS].values, name),
            feas_label(df),
            df["burr_evaluated"].values,
        )

    Xs_L, ys_L, _    = src_data(src_largest)
    Xs_S, ys_S, bs_S = src_data(src_similar)

    # ---- Active labelling: pretrain on most-similar source, query uncertain targets ----
    pretrain = make_clf().fit(Xs_S, ys_S)
    idx_lab  = pick_active_labels_idx(Xt_all_n, pretrain, n_labels=n_lab)
    idx_te   = np.setdiff1d(np.arange(len(yt_full)), idx_lab)
    if len(idx_te) < 5:
        return None
    Xt_lab_n = Xt_all_n[idx_lab]; yt_lab = yt_full[idx_lab]; bt_lab = burr_t[idx_lab]
    Xt_te_n  = Xt_all_n[idx_te];  yt_te  = yt_full[idx_te]

    def acc_clf(X_tr, y_tr):
        if X_tr is None or len(X_tr) < 2 or len(np.unique(y_tr)) < 2:
            return np.nan
        return accuracy_score(yt_te, make_clf().fit(X_tr, y_tr).predict(Xt_te_n))

    out = {
        "source_largest":      src_largest,
        "source_similar":      src_similar,
        "sim_cost_top1":       ranked[0][1],
        "n_active_feasible":   int(yt_lab.sum()),  # diagnostic: did the active pick span both classes?
        "target_only":         acc_clf(Xt_lab_n, yt_lab),
        "source_largest_only": acc_clf(Xs_L, ys_L),
        "source_similar_only": acc_clf(Xs_S, ys_S),
    }

    # ---- OT methods (identical to v2) ----
    try:
        with warnings.catch_warnings(), redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            warnings.simplefilter("ignore")
            ot_l1l2 = SinkhornL1l2Transport(reg_e=sinkhorn_reg, reg_cl=1.0, max_iter=20)
            ot_l1l2.fit(Xs=Xs_S, ys=ys_S, Xt=Xt_lab_n, yt=yt_lab)
            Xs_l1l2 = ot_l1l2.transform(Xs=Xs_S)
        out["classreg_OT_similar"] = acc_clf(Xs_l1l2, ys_S)
    except Exception:
        out["classreg_OT_similar"] = np.nan

    top_sources = [s for s, _ in ranked[:top_k]]
    pooled      = [src_data(s) for s in top_sources]
    Xs_pool = np.concatenate([X for X, _, _ in pooled])
    ys_pool = np.concatenate([y for _, y, _ in pooled])
    try:
        with warnings.catch_warnings(), redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            warnings.simplefilter("ignore")
            ot_multi = SinkhornL1l2Transport(reg_e=sinkhorn_reg, reg_cl=1.0, max_iter=20)
            ot_multi.fit(Xs=Xs_pool, ys=ys_pool, Xt=Xt_lab_n, yt=yt_lab)
            Xs_multi = ot_multi.transform(Xs=Xs_pool)
        out["multisource_OT"] = acc_clf(Xs_multi, ys_pool)
    except Exception:
        out["multisource_OT"] = np.nan

    try:
        Xs_aug = np.concatenate([
            augment_with_burr(X_s, b_s, name)
            for (X_s, _, b_s), name in zip(pooled, top_sources)
        ])
        Xt_lab_aug = augment_with_burr(Xt_lab_n, bt_lab, target_name)
        with warnings.catch_warnings(), redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            warnings.simplefilter("ignore")
            ot_jdot = SinkhornL1l2Transport(reg_e=sinkhorn_reg, reg_cl=1.0, max_iter=20)
            ot_jdot.fit(Xs=Xs_aug, ys=ys_pool, Xt=Xt_lab_aug, yt=yt_lab)
            Xs_jdot_aug = ot_jdot.transform(Xs=Xs_aug)
        out["multisource_OT_burr"] = acc_clf(Xs_jdot_aug[:, :-1], ys_pool)
    except Exception:
        out["multisource_OT_burr"] = np.nan

    return out


In [221]:
# --- Run the active-labelled pipeline once per target ---
rows_v3, skipped_v3 = [], []
for name in task_df_dict:
    res = run_ot_analysis_v3(name)
    if res is None:
        skipped_v3.append(name)
        continue
    rows_v3.append({"target_task": name, **res})

results_df_v3 = pd.DataFrame(rows_v3)
print(f"Completed: {len(results_df_v3)} / {len(task_df_dict)} tasks")
print(
    f"Active label balance — # feasible in the 15-point pick:  "
    f"min={results_df_v3['n_active_feasible'].min()}, "
    f"median={results_df_v3['n_active_feasible'].median():.1f}, "
    f"max={results_df_v3['n_active_feasible'].max()}"
)

methods_v3 = methods_v2
labels_v3  = [lab + ", active" for lab in labels_v2]
colors_v3  = colors_v2

# 1) Boxplot
fig = go.Figure()
for m, lab, c in zip(methods_v3, labels_v3, colors_v3):
    fig.add_trace(go.Box(
        y=results_df_v3[m].dropna(), name=lab, marker_color=c,
        boxpoints="all", jitter=0.3, pointpos=-1.6,
    ))
fig.update_layout(
    title=f"Active labelling — accuracy across {len(results_df_v3)} target tasks",
    yaxis_title="Accuracy", yaxis=dict(range=[0, 1.05]),
    width=950, height=500,
)
fig.show()

# 2) Heatmap of v3 (winner per row outlined)
short_names = [n if len(n) <= 36 else n[:33] + "..." for n in results_df_v3["target_task"]]
heat = results_df_v3[methods_v3].values
heat_for_argmax = np.where(np.isnan(heat), -np.inf, heat)
best_col = np.argmax(heat_for_argmax, axis=1)
text_matrix = [
    [
        ("" if np.isnan(v)
         else (f"<b>{v:.2f}</b>" if j == best_col[i] else f"{v:.2f}"))
        for j, v in enumerate(row)
    ]
    for i, row in enumerate(heat)
]
fig = go.Figure(data=go.Heatmap(
    z=heat, x=labels_v3, y=short_names,
    text=text_matrix, texttemplate="%{text}", textfont=dict(size=10),
    colorscale="RdYlGn", zmin=0, zmax=1, colorbar=dict(title="accuracy"),
))
for i, j in enumerate(best_col):
    fig.add_shape(
        type="rect", xref="x", yref="y",
        x0=j - 0.5, x1=j + 0.5, y0=i - 0.5, y1=i + 0.5,
        line=dict(color="black", width=2.5),
        fillcolor="rgba(0,0,0,0)", layer="above",
    )
fig.update_layout(
    title="Accuracy heatmap — active labelling (winner per row outlined)",
    width=950, height=max(450, 24 * len(results_df_v3)),
    yaxis=dict(autorange="reversed"),
)
fig.show()

# 3) Δ heatmap: v3 (active) minus v2 (random)  — directly shows the active-labelling effect
common = sorted(set(results_df_v2["target_task"]) & set(results_df_v3["target_task"]))
v2_lkp = results_df_v2.set_index("target_task")
v3_lkp = results_df_v3.set_index("target_task")
delta = v3_lkp.loc[common, methods_v2].values - v2_lkp.loc[common, methods_v2].values
short_common = [n if len(n) <= 36 else n[:33] + "..." for n in common]
delta_text = [[("" if np.isnan(v) else f"{v:+.2f}") for v in row] for row in delta]
max_abs = max(np.nanmax(np.abs(delta)), 0.05)
fig = go.Figure(data=go.Heatmap(
    z=delta, x=labels_v2, y=short_common,
    text=delta_text, texttemplate="%{text}", textfont=dict(size=10),
    colorscale="RdBu", zmid=0, zmin=-max_abs, zmax=max_abs,
    colorbar=dict(title="Δ accuracy"),
))
fig.update_layout(
    title="Δ accuracy — active (v3) minus random (v2) per (target, method)<br>"
          "<sup>Blue = active labelling helped, red = it hurt.</sup>",
    width=950, height=max(450, 24 * len(common)),
    yaxis=dict(autorange="reversed"),
)
fig.show()

# 4) Summary table + mean delta per method
summary = results_df_v3[methods_v3].agg(["mean", "median", "std"]).T.round(3)
summary.index = labels_v3
summary["wins_vs_target_only"] = [
    (results_df_v3[m] > results_df_v3["target_only"]).sum() for m in methods_v3
]
summary["wins_vs_largest"] = [
    (results_df_v3[m] > results_df_v3["source_largest_only"]).sum() for m in methods_v3
]
summary["best_count"] = (
    results_df_v3[methods_v3].rank(axis=1, ascending=False, method="min") == 1
).sum().values
print("\nPer-method summary (active labelling):\n")
print(summary)

mean_delta = pd.Series(np.nanmean(delta, axis=0).round(3), index=labels_v2,
                       name="mean Δ vs random")
print("\nMean accuracy gain from active labelling (positive = active helps):\n")
print(mean_delta)


Completed: 23 / 27 tasks
Active label balance — # feasible in the 15-point pick:  min=2, median=11.0, max=15



Per-method summary (active labelling):

                                          mean  median    std  \
target-only (GBM), active                0.699   0.714  0.155   
source-only (largest, GBM), active       0.854   0.864  0.094   
source-only (similar, GBM), active       0.860   0.840  0.110   
class-reg OT (similar, GBM), active      0.770   0.760  0.092   
multi-source OT (top-3 similar), active  0.770   0.812  0.145   
multi-source OT + burr (JDOT), active    0.787   0.800  0.138   

                                         wins_vs_target_only  wins_vs_largest  \
target-only (GBM), active                                  0                4   
source-only (largest, GBM), active                        16                0   
source-only (similar, GBM), active                        16               13   
class-reg OT (similar, GBM), active                       11                5   
multi-source OT (top-3 similar), active                   12                7   
multi-source OT +

---

## Step 10 — Multi-Source TrAdaBoost (Yao & Doretto)

Direct port of `mstradaboost.m` / `mstrpredict.m` in this directory. Algorithm:

1. **M = 10 boosting rounds.** Each round trains an RBF-SVM weak learner per
   candidate source on `(weighted source_k ∪ weighted target)`, then *picks the
   weak learner with smallest weighted error on the target portion*.
2. **Weight updates** after each round:
   - Source weights *decay* on errors (`ws *= exp(-aₛ · |pred − y|)`) — pushes
     out source points that don't transfer cleanly.
   - Target weights *grow* on errors (`wt *= exp(α_t · |pred − y|)`) — standard
     AdaBoost focus on hard targets.
3. **Prediction**: `sign(Σₜ αₜ · fₜ(x))`.

We re-use the **top-3 most-similar sources** from Step 8's similarity ranking and
the **random-labelling** target split (matches v2 so the comparison is direct).
RBF-SVM with `gamma="scale"` is the closest sklearn analogue of the MATLAB
`-t 2` kernel used in the original.


In [222]:
# --- Multi-Source TrAdaBoost (port of mstradaboost.m / mstrpredict.m) ---
from sklearn.svm import SVC


def ms_tradaboost_fit(sources, X_t, y_t, M=10):
    """Multi-source TrAdaBoost (Yao & Doretto).

    sources : list of (X_s, y_s) tuples. Labels in {0, 1}.
    X_t, y_t: target training data, labels in {0, 1}.
    Returns list of (svm_model, alpha) — the M weak learners.
    """
    def to_pm(y):
        return np.where(np.asarray(y) == 1, 1.0, -1.0)

    sources_pm = [(np.asarray(X_s, dtype=float), to_pm(y_s)) for X_s, y_s in sources]
    X_t = np.asarray(X_t, dtype=float)
    y_t_pm = to_pm(y_t)

    N = len(sources_pm)
    ns = sum(len(y) for _, y in sources_pm)
    m  = len(y_t_pm)
    if N == 0 or m == 0:
        return []

    ws = [np.ones(len(y)) for _, y in sources_pm]
    wt = np.ones(m)
    sW = wt.sum()                                          # frozen normaliser (matches MATLAB)
    as_const = np.log(1 + np.sqrt(2 * np.log(max(ns / M, 1.0001)))) / 2.0

    hyps = []
    for _ in range(M):
        best_err   = np.inf
        best_model = None

        for k in range(N):
            X_s, y_s = sources_pm[k]
            Xc = np.vstack([X_s, X_t])
            yc = np.concatenate([y_s, y_t_pm])
            wc = np.concatenate([ws[k], wt])
            # Both classes must be present after weighting; if a class collapsed
            # to zero weight, skip this candidate.
            if len(np.unique(yc[wc > 0])) < 2:
                continue
            try:
                clf = SVC(kernel="rbf", gamma="scale", C=1.0)
                clf.fit(Xc, yc, sample_weight=wc)
            except Exception:
                continue
            preds_t = clf.predict(X_t)
            err = float((wt * (preds_t != y_t_pm)).sum() / sW)
            if err < best_err:
                best_err   = err
                best_model = clf

        if best_model is None:
            break  # no usable weak learner this round; bail with what we have

        err_t = float(np.clip(best_err, 0.001, 0.499))
        alpha = 0.5 * np.log((1 - err_t) / err_t)
        hyps.append((best_model, alpha))

        # Source weights: DECAY on errors  (clip exponent for numerical safety)
        for k in range(N):
            X_s, y_s = sources_pm[k]
            preds_s = best_model.predict(X_s)
            err_s   = np.abs(preds_s - y_s)               # {0, 2} on ±1 labels
            ws[k]   = ws[k] * np.exp(np.clip(-as_const * err_s, -50, 50))

        # Target weights: GROW on errors
        preds_t   = best_model.predict(X_t)
        err_t_arr = np.abs(preds_t - y_t_pm)
        wt        = wt * np.exp(np.clip(alpha * err_t_arr, -50, 50))

    return hyps


def ms_tradaboost_predict(hyps, X):
    """Aggregate ±1 predictions weighted by alpha; return 0/1 labels."""
    if not hyps:
        return np.zeros(len(X), dtype=int)
    X = np.asarray(X, dtype=float)
    score = np.zeros(len(X))
    for clf, alpha in hyps:
        score = score + alpha * clf.predict(X)
    pred_pm = np.sign(score)
    pred_pm[pred_pm == 0] = 1                              # tie-break to positive class
    return ((pred_pm + 1) / 2).astype(int)


def run_mstradaboost(target_name, train_size=15, top_k=3, M=10, random_state=0):
    """Random-label split (matches v2) → top-3 similar sources → MS-TrAdaBoost."""
    df_t = task_df_dict[target_name]
    yt_full = feas_label(df_t)
    if (yt_full == 1).sum() < 2 or (yt_full == 0).sum() < 2:
        return None

    Xt_all_n = to_unit_cube(df_t[FEATURE_COLS].values, target_name)
    n_lab = min(train_size, len(yt_full) - 5)
    if n_lab < 4:
        return None
    try:
        idx_lab, idx_te = train_test_split(
            np.arange(len(yt_full)), train_size=n_lab,
            stratify=yt_full, random_state=random_state,
        )
    except ValueError:
        return None
    Xt_lab = Xt_all_n[idx_lab]; yt_lab = yt_full[idx_lab]
    Xt_te  = Xt_all_n[idx_te];  yt_te  = yt_full[idx_te]

    ranked = rank_sources_by_similarity(target_name, Xt_all_n)
    if not ranked:
        return None
    top_sources = [s for s, _ in ranked[:top_k]]
    sources = []
    for s in top_sources:
        df = task_df_dict[s]
        sources.append((to_unit_cube(df[FEATURE_COLS].values, s), feas_label(df)))

    try:
        hyps = ms_tradaboost_fit(sources, Xt_lab, yt_lab, M=M)
        if not hyps:
            return None
        preds = ms_tradaboost_predict(hyps, Xt_te)
        return {
            "target_task":     target_name,
            "mstradaboost":    float(accuracy_score(yt_te, preds)),
            "n_weak_learners": len(hyps),
        }
    except Exception:
        return None


In [223]:
# --- Run MS-TrAdaBoost rotation and merge with v2 results for unified comparison ---
mstr_rows = []
for name in task_df_dict:
    r = run_mstradaboost(name)
    if r is not None:
        mstr_rows.append(r)

mstr_df = pd.DataFrame(mstr_rows)
print(f"Completed: {len(mstr_df)} / {len(task_df_dict)} tasks")
print(f"Avg weak learners per target: {mstr_df['n_weak_learners'].mean():.1f} / 10  "
      f"(min={mstr_df['n_weak_learners'].min()}, max={mstr_df['n_weak_learners'].max()})")

# Combine with v2's results so MS-TrAdaBoost is just one more column on the same chart
combined = results_df_v2.merge(
    mstr_df[["target_task", "mstradaboost"]], on="target_task", how="left",
)

all_methods = methods_v2 + ["mstradaboost"]
all_labels  = labels_v2  + ["MS-TrAdaBoost (top-3 similar)"]
all_colors  = colors_v2  + ["#444444"]

# 1) Boxplot
fig = go.Figure()
for m, lab, c in zip(all_methods, all_labels, all_colors):
    fig.add_trace(go.Box(
        y=combined[m].dropna(), name=lab, marker_color=c,
        boxpoints="all", jitter=0.3, pointpos=-1.6,
    ))
fig.update_layout(
    title=f"All methods (random labelling) including MS-TrAdaBoost — {len(combined)} target tasks",
    yaxis_title="Accuracy", yaxis=dict(range=[0, 1.05]),
    width=1000, height=500,
)
fig.show()

# 2) Heatmap with winner per row outlined
short_names = [n if len(n) <= 36 else n[:33] + "..." for n in combined["target_task"]]
heat = combined[all_methods].values
heat_for_argmax = np.where(np.isnan(heat), -np.inf, heat)
best_col = np.argmax(heat_for_argmax, axis=1)
text_matrix = [
    [
        ("" if np.isnan(v)
         else (f"<b>{v:.2f}</b>" if j == best_col[i] else f"{v:.2f}"))
        for j, v in enumerate(row)
    ]
    for i, row in enumerate(heat)
]
fig = go.Figure(data=go.Heatmap(
    z=heat, x=all_labels, y=short_names,
    text=text_matrix, texttemplate="%{text}", textfont=dict(size=10),
    colorscale="RdYlGn", zmin=0, zmax=1, colorbar=dict(title="accuracy"),
))
for i, j in enumerate(best_col):
    fig.add_shape(
        type="rect", xref="x", yref="y",
        x0=j - 0.5, x1=j + 0.5, y0=i - 0.5, y1=i + 0.5,
        line=dict(color="black", width=2.5),
        fillcolor="rgba(0,0,0,0)", layer="above",
    )
fig.update_layout(
    title="Accuracy heatmap — all methods + MS-TrAdaBoost (winner per row outlined)",
    width=1000, height=max(450, 24 * len(combined)),
    yaxis=dict(autorange="reversed"),
)
fig.show()

# 3) Summary
summary = combined[all_methods].agg(["mean", "median", "std"]).T.round(3)
summary.index = all_labels
summary["wins_vs_target_only"] = [
    (combined[m] > combined["target_only"]).sum() for m in all_methods
]
summary["wins_vs_largest"] = [
    (combined[m] > combined["source_largest_only"]).sum() for m in all_methods
]
summary["best_count"] = (
    combined[all_methods].rank(axis=1, ascending=False, method="min") == 1
).sum().values
print("\nPer-method summary (with MS-TrAdaBoost):\n")
print(summary)


Completed: 23 / 27 tasks
Avg weak learners per target: 10.0 / 10  (min=10, max=10)



Per-method summary (with MS-TrAdaBoost):

                                  mean  median    std  wins_vs_target_only  \
target-only (GBM)                0.870   0.890  0.073                    0   
source-only (largest, GBM)       0.827   0.844  0.106                    9   
source-only (similar, GBM)       0.813   0.800  0.120                    8   
class-reg OT (similar, GBM)      0.761   0.766  0.112                    4   
multi-source OT (top-3 similar)  0.707   0.735  0.122                    3   
multi-source OT + burr (JDOT)    0.803   0.826  0.106                    6   
MS-TrAdaBoost (top-3 similar)    0.866   0.880  0.068                    9   

                                 wins_vs_largest  best_count  
target-only (GBM)                             13          10  
source-only (largest, GBM)                     0           5  
source-only (similar, GBM)                    12           6  
class-reg OT (similar, GBM)                    7           0  
multi-source OT (

---

## Step 11 — DeepJDOT (Damodaran et al., ECCV 2018)

End-to-end deep version of JDOT. A small MLP encoder `g` and classifier head `f`
are trained jointly with two losses:

- **Supervised CE** on the 15 labelled target points (anchors `f` to ground truth).
- **JDOT loss** = `Σᵢⱼ γᵢⱼ · ( α·||g(xₛᵢ) − g(xₜⱼ)||² + β·CE(yₛᵢ, f(g(xₜⱼ))) )`,
  where γ is the Sinkhorn OT plan computed on the *same* joint cost.

The OT plan is recomputed every `ot_every` epochs (kept fixed in between to make
gradient flow tractable). The OT uses **all target features** — test-set points
contribute features only, no label leakage — so it's semi-supervised.

Caveats for this dataset:
- Only 3 input features and ~50–300 points per task, so the network is
  intentionally tiny (`hidden=16`). With anything bigger the MLP will simply
  memorise the 15 labelled target points.
- Same source pool as Steps 8/10 (top-3 most-similar) and same random label
  split as v2 — so the result merges directly into the unified comparison.


In [224]:
# --- DeepJDOT (Damodaran et al. 2018) ---
import torch
import torch.nn as nn
import torch.nn.functional as F


class _DeepJDOTNet(nn.Module):
    def __init__(self, in_dim=3, hidden=16, n_classes=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.classifier(z), z


def train_deepjdot(
    Xs, ys, Xt_lab, yt_lab, Xt_all,
    n_epochs=100, ot_every=10,
    alpha_feat=0.5, alpha_label=1.0,
    lr=1e-3, hidden=16, sinkhorn_reg=0.05, seed=0,
):
    """Train a DeepJDOT model. Returns the trained nn.Module."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    Xs_t     = torch.tensor(Xs,     dtype=torch.float32)
    ys_t     = torch.tensor(ys,     dtype=torch.long)
    Xt_lab_t = torch.tensor(Xt_lab, dtype=torch.float32)
    yt_lab_t = torch.tensor(yt_lab, dtype=torch.long)
    Xt_all_t = torch.tensor(Xt_all, dtype=torch.float32)

    net = _DeepJDOTNet(in_dim=Xs.shape[1], hidden=hidden, n_classes=2).to(torch.float32)
    opt = torch.optim.Adam(net.parameters(), lr=lr)

    n_s, n_t = len(Xs_t), len(Xt_all_t)
    a = np.ones(n_s) / n_s
    b = np.ones(n_t) / n_t
    gamma = torch.zeros(n_s, n_t, dtype=torch.float32)

    for epoch in range(n_epochs):
        # Periodically recompute the OT plan with the current encoder/classifier
        if epoch % ot_every == 0:
            net.eval()
            with torch.no_grad():
                _, z_s          = net(Xs_t)
                logits_t, z_t   = net(Xt_all_t)
                proba_t         = F.softmax(logits_t, dim=1)
                # Joint cost: feature distance in embedding space + CE(y_s, proba_t)
                feat_cost  = ((z_s.unsqueeze(1) - z_t.unsqueeze(0)) ** 2).sum(-1)
                # label_cost[i, j] = -log(proba_t[j, y_s[i]])
                label_cost = -torch.log(proba_t[:, ys_t].t() + 1e-10)
                cost       = alpha_feat * feat_cost + alpha_label * label_cost
                cost_np    = cost.cpu().numpy()
                cost_np    = cost_np / max(cost_np.max(), 1e-12)
                gamma_np   = ot.sinkhorn(a, b, cost_np, reg=sinkhorn_reg, numItermax=100)
                gamma      = torch.tensor(gamma_np, dtype=torch.float32)
            net.train()

        opt.zero_grad()

        # Supervised loss on the labelled target points
        logits_lab, _ = net(Xt_lab_t)
        loss_sup = F.cross_entropy(logits_lab, yt_lab_t)

        # JDOT loss with the (fixed) OT plan from the most recent recompute
        _, z_s              = net(Xs_t)
        logits_t_all, z_t   = net(Xt_all_t)
        proba_t_all         = F.softmax(logits_t_all, dim=1)
        feat_cost           = ((z_s.unsqueeze(1) - z_t.unsqueeze(0)) ** 2).sum(-1)
        label_cost          = -torch.log(proba_t_all[:, ys_t].t() + 1e-10)
        joint_cost          = alpha_feat * feat_cost + alpha_label * label_cost
        loss_ot             = (gamma * joint_cost).sum()

        (loss_sup + loss_ot).backward()
        opt.step()

    return net


def predict_deepjdot(net, X):
    net.eval()
    with torch.no_grad():
        logits, _ = net(torch.tensor(X, dtype=torch.float32))
        return logits.argmax(dim=1).cpu().numpy()


def run_deepjdot(target_name, train_size=15, top_k=3,
                 n_epochs=100, ot_every=10, hidden=16, seed=0):
    """Random-label split (matches v2) → top-3 similar sources pooled → DeepJDOT."""
    df_t = task_df_dict[target_name]
    yt_full = feas_label(df_t).astype(int)
    if (yt_full == 1).sum() < 2 or (yt_full == 0).sum() < 2:
        return None

    Xt_all_n = to_unit_cube(df_t[FEATURE_COLS].values, target_name)
    n_lab = min(train_size, len(yt_full) - 5)
    if n_lab < 4:
        return None
    try:
        idx_lab, idx_te = train_test_split(
            np.arange(len(yt_full)), train_size=n_lab,
            stratify=yt_full, random_state=seed,
        )
    except ValueError:
        return None
    Xt_lab = Xt_all_n[idx_lab]; yt_lab = yt_full[idx_lab]
    Xt_te  = Xt_all_n[idx_te];  yt_te  = yt_full[idx_te]

    ranked = rank_sources_by_similarity(target_name, Xt_all_n)
    if not ranked:
        return None
    top_sources = [s for s, _ in ranked[:top_k]]
    Xs_pool, ys_pool = [], []
    for s in top_sources:
        df = task_df_dict[s]
        Xs_pool.append(to_unit_cube(df[FEATURE_COLS].values, s))
        ys_pool.append(feas_label(df).astype(int))
    Xs_pool = np.concatenate(Xs_pool)
    ys_pool = np.concatenate(ys_pool)

    try:
        net = train_deepjdot(
            Xs_pool, ys_pool, Xt_lab, yt_lab, Xt_all_n,
            n_epochs=n_epochs, ot_every=ot_every, hidden=hidden, seed=seed,
        )
        preds = predict_deepjdot(net, Xt_te)
        return {"target_task": target_name, "deepjdot": float(accuracy_score(yt_te, preds))}
    except Exception as e:
        return {"target_task": target_name, "deepjdot": np.nan, "error": str(e)}


In [225]:
# --- Run DeepJDOT rotation and produce the unified comparison (v2 + MS-TrAdaBoost + DeepJDOT) ---
djdot_rows = []
for name in task_df_dict:
    r = run_deepjdot(name)
    if r is not None:
        djdot_rows.append(r)
djdot_df = pd.DataFrame(djdot_rows)
print(f"Completed: {len(djdot_df)} / {len(task_df_dict)} tasks")
if "error" in djdot_df.columns:
    n_err = djdot_df["error"].notna().sum()
    if n_err:
        print(f"  ({n_err} tasks errored — see `djdot_df` for messages)")

# Merge: v2 (6 columns) + MS-TrAdaBoost (1) + DeepJDOT (1) = 8 methods
combined_full = combined.merge(
    djdot_df[["target_task", "deepjdot"]], on="target_task", how="left",
)

all_methods_full = all_methods + ["deepjdot"]
all_labels_full  = all_labels  + ["DeepJDOT (top-3 similar)"]
all_colors_full  = all_colors  + ["#1f77b4"]

# 1) Boxplot
fig = go.Figure()
for m, lab, c in zip(all_methods_full, all_labels_full, all_colors_full):
    fig.add_trace(go.Box(
        y=combined_full[m].dropna(), name=lab, marker_color=c,
        boxpoints="all", jitter=0.3, pointpos=-1.6,
    ))
fig.update_layout(
    title=f"All methods — including MS-TrAdaBoost and DeepJDOT  ({len(combined_full)} target tasks)",
    yaxis_title="Accuracy", yaxis=dict(range=[0, 1.05]),
    width=1050, height=520,
)
fig.show()

# 2) Heatmap with winner per row outlined
short_names = [n if len(n) <= 36 else n[:33] + "..." for n in combined_full["target_task"]]
heat = combined_full[all_methods_full].values
heat_for_argmax = np.where(np.isnan(heat), -np.inf, heat)
best_col = np.argmax(heat_for_argmax, axis=1)
text_matrix = [
    [
        ("" if np.isnan(v)
         else (f"<b>{v:.2f}</b>" if j == best_col[i] else f"{v:.2f}"))
        for j, v in enumerate(row)
    ]
    for i, row in enumerate(heat)
]
fig = go.Figure(data=go.Heatmap(
    z=heat, x=all_labels_full, y=short_names,
    text=text_matrix, texttemplate="%{text}", textfont=dict(size=10),
    colorscale="RdYlGn", zmin=0, zmax=1, colorbar=dict(title="accuracy"),
))
for i, j in enumerate(best_col):
    fig.add_shape(
        type="rect", xref="x", yref="y",
        x0=j - 0.5, x1=j + 0.5, y0=i - 0.5, y1=i + 0.5,
        line=dict(color="black", width=2.5),
        fillcolor="rgba(0,0,0,0)", layer="above",
    )
fig.update_layout(
    title="Accuracy heatmap — all 8 methods (winner per row outlined)",
    width=1050, height=max(450, 24 * len(combined_full)),
    yaxis=dict(autorange="reversed"),
)
fig.show()

# 3) Summary
summary = combined_full[all_methods_full].agg(["mean", "median", "std"]).T.round(3)
summary.index = all_labels_full
summary["wins_vs_target_only"] = [
    (combined_full[m] > combined_full["target_only"]).sum() for m in all_methods_full
]
summary["wins_vs_largest"] = [
    (combined_full[m] > combined_full["source_largest_only"]).sum() for m in all_methods_full
]
summary["best_count"] = (
    combined_full[all_methods_full].rank(axis=1, ascending=False, method="min") == 1
).sum().values
print("\nPer-method summary (all 8 methods):\n")
print(summary)


Completed: 23 / 27 tasks



Per-method summary (all 8 methods):

                                  mean  median    std  wins_vs_target_only  \
target-only (GBM)                0.870   0.890  0.073                    0   
source-only (largest, GBM)       0.827   0.844  0.106                    9   
source-only (similar, GBM)       0.813   0.800  0.120                    8   
class-reg OT (similar, GBM)      0.761   0.766  0.112                    4   
multi-source OT (top-3 similar)  0.707   0.735  0.122                    3   
multi-source OT + burr (JDOT)    0.803   0.826  0.106                    6   
MS-TrAdaBoost (top-3 similar)    0.866   0.880  0.068                    9   
DeepJDOT (top-3 similar)         0.809   0.838  0.103                    6   

                                 wins_vs_largest  best_count  
target-only (GBM)                             13          10  
source-only (largest, GBM)                     0           5  
source-only (similar, GBM)                    12           6  
class-r

---

## Step 12 — Ensemble of the four top-of-class methods

Combine the four methods that survived Steps 8/10/11 — each genuinely different
in *what it transfers* and *how* — into a single prediction:

| Member | Mechanism |
|---|---|
| `multi-source OT (top-3)`        | class-regularised OT in feature space, GBM head |
| `multi-source OT + burr (JDOT)`  | same OT but the cost includes burr — feasibility *and* burr similarity |
| `MS-TrAdaBoost (top-3)`          | boosted RBF-SVM weak learners with reweighted source/target |
| `DeepJDOT (top-3)`               | learned embedding + JDOT alignment (deep) |

Three ensemble strategies, all computed on the same targets and the same
random label split as v2 (so member accuracies match the columns you've already
seen in Steps 8/10/11):

- **hard majority** — each member votes `{0, 1}`; predict `1` if at least half
  vote `1` (ties → `1`, the majority class).
- **hard strict** — predict `1` only on a *strict* majority (ties → `0`,
  conservative).
- **soft average** — average `P(feasible)` across members, threshold at `0.5`.
  MS-TrAdaBoost's score is sigmoid-mapped to a probability since it doesn't
  produce one natively; the others use `predict_proba` / softmax.


In [226]:
# --- Ensemble of multi-source OT, OT + burr, MS-TrAdaBoost, DeepJDOT ---
def run_ensemble(target_name, train_size=15, top_k=3, M=10, n_epochs=100, random_state=0):
    """Train all four members on the same v2-style split and combine."""
    df_t = task_df_dict[target_name]
    yt_full = feas_label(df_t).astype(int)
    if (yt_full == 1).sum() < 2 or (yt_full == 0).sum() < 2:
        return None

    Xt_all_n = to_unit_cube(df_t[FEATURE_COLS].values, target_name)
    burr_t = df_t["burr_evaluated"].values
    n_lab = min(train_size, len(yt_full) - 5)
    if n_lab < 4:
        return None
    try:
        idx_lab, idx_te = train_test_split(
            np.arange(len(yt_full)), train_size=n_lab,
            stratify=yt_full, random_state=random_state,
        )
    except ValueError:
        return None
    Xt_lab = Xt_all_n[idx_lab]; yt_lab = yt_full[idx_lab]; bt_lab = burr_t[idx_lab]
    Xt_te  = Xt_all_n[idx_te];  yt_te  = yt_full[idx_te]

    ranked = rank_sources_by_similarity(target_name, Xt_all_n)
    if not ranked:
        return None
    top_sources = [s for s, _ in ranked[:top_k]]
    pooled = []
    for s in top_sources:
        df = task_df_dict[s]
        pooled.append((
            to_unit_cube(df[FEATURE_COLS].values, s),
            feas_label(df).astype(int),
            df["burr_evaluated"].values,
        ))
    Xs_pool = np.concatenate([X for X, _, _ in pooled])
    ys_pool = np.concatenate([y for _, y, _ in pooled])

    preds, proba = {}, {}

    # 1) multi-source OT
    try:
        with warnings.catch_warnings(), redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            warnings.simplefilter("ignore")
            ot_m = SinkhornL1l2Transport(reg_e=0.05, reg_cl=1.0, max_iter=20)
            ot_m.fit(Xs=Xs_pool, ys=ys_pool, Xt=Xt_lab, yt=yt_lab)
            Xs_t = ot_m.transform(Xs=Xs_pool)
        clf = make_clf().fit(Xs_t, ys_pool)
        preds["multisource_OT"] = clf.predict(Xt_te).astype(int)
        proba["multisource_OT"] = clf.predict_proba(Xt_te)[:, 1]
    except Exception:
        pass

    # 2) multi-source OT + burr (JDOT flavour)
    try:
        Xs_aug = np.concatenate([
            augment_with_burr(X_s, b_s, name)
            for (X_s, _, b_s), name in zip(pooled, top_sources)
        ])
        Xt_lab_aug = augment_with_burr(Xt_lab, bt_lab, target_name)
        with warnings.catch_warnings(), redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            warnings.simplefilter("ignore")
            ot_j = SinkhornL1l2Transport(reg_e=0.05, reg_cl=1.0, max_iter=20)
            ot_j.fit(Xs=Xs_aug, ys=ys_pool, Xt=Xt_lab_aug, yt=yt_lab)
            Xs_t = ot_j.transform(Xs=Xs_aug)
        clf = make_clf().fit(Xs_t[:, :-1], ys_pool)
        preds["multisource_OT_burr"] = clf.predict(Xt_te).astype(int)
        proba["multisource_OT_burr"] = clf.predict_proba(Xt_te)[:, 1]
    except Exception:
        pass

    # 3) MS-TrAdaBoost (sigmoid-map score → P(feasible))
    try:
        sources = [(X, y) for (X, y, _) in pooled]
        hyps = ms_tradaboost_fit(sources, Xt_lab, yt_lab, M=M)
        if hyps:
            score = np.zeros(len(Xt_te))
            for c, a in hyps:
                score = score + a * c.predict(Xt_te.astype(float))
            preds["mstradaboost"] = (np.sign(score) > 0).astype(int)
            alpha_sum = sum(a for _, a in hyps)
            scale = max(alpha_sum, 1e-6)
            proba["mstradaboost"] = 1.0 / (1.0 + np.exp(-2.0 * score / scale))
    except Exception:
        pass

    # 4) DeepJDOT
    try:
        net = train_deepjdot(
            Xs_pool, ys_pool, Xt_lab, yt_lab, Xt_all_n,
            n_epochs=n_epochs, ot_every=10, hidden=16, seed=random_state,
        )
        net.eval()
        with torch.no_grad():
            logits, _ = net(torch.tensor(Xt_te, dtype=torch.float32))
            p1 = F.softmax(logits, dim=1).cpu().numpy()[:, 1]
        preds["deepjdot"] = (p1 >= 0.5).astype(int)
        proba["deepjdot"] = p1
    except Exception:
        pass

    if len(preds) < 2:
        return None  # need at least 2 members to ensemble

    # Hard votes
    P = np.stack(list(preds.values()))           # (n_methods, n_test)
    n_m = P.shape[0]
    vote_sum = P.sum(axis=0)
    hard_majority = (vote_sum >= (n_m / 2.0)).astype(int)        # ties → 1
    hard_strict   = (vote_sum >  (n_m / 2.0)).astype(int)        # ties → 0

    # Soft average
    Q = np.stack(list(proba.values()))
    avg_proba = Q.mean(axis=0)
    soft_avg  = (avg_proba >= 0.5).astype(int)

    out = {
        "target_task":  target_name,
        "n_members":    n_m,
        "ens_majority": float(accuracy_score(yt_te, hard_majority)),
        "ens_strict":   float(accuracy_score(yt_te, hard_strict)),
        "ens_soft_avg": float(accuracy_score(yt_te, soft_avg)),
    }
    for name, p in preds.items():
        out[f"member_{name}"] = float(accuracy_score(yt_te, p))
    return out


In [227]:
# --- Run the ensemble rotation and visualise members vs combined predictions ---
ens_rows = []
for name in task_df_dict:
    r = run_ensemble(name)
    if r is not None:
        ens_rows.append(r)

ens_df = pd.DataFrame(ens_rows)
print(f"Completed: {len(ens_df)} / {len(task_df_dict)} tasks")
print(f"Avg # members per ensemble: {ens_df['n_members'].mean():.2f} / 4")

ens_methods = [
    "member_multisource_OT",
    "member_multisource_OT_burr",
    "member_mstradaboost",
    "member_deepjdot",
    "ens_majority",
    "ens_strict",
    "ens_soft_avg",
]
ens_labels = [
    "multi-source OT",
    "multi-source OT + burr",
    "MS-TrAdaBoost",
    "DeepJDOT",
    "ENSEMBLE — hard majority",
    "ENSEMBLE — hard strict",
    "ENSEMBLE — soft average",
]
ens_colors = [
    "seagreen", "mediumpurple", "#444444", "#1f77b4",
    "#d62728", "#ff7f0e", "#2ca02c",
]

# 1) Boxplot
fig = go.Figure()
for m, lab, c in zip(ens_methods, ens_labels, ens_colors):
    fig.add_trace(go.Box(
        y=ens_df[m].dropna(), name=lab, marker_color=c,
        boxpoints="all", jitter=0.3, pointpos=-1.6,
    ))
fig.update_layout(
    title=f"Ensemble vs members — {len(ens_df)} target tasks",
    yaxis_title="Accuracy", yaxis=dict(range=[0, 1.05]),
    width=1050, height=520,
)
fig.show()

# 2) Heatmap with winner per row outlined
short_names = [n if len(n) <= 36 else n[:33] + "..." for n in ens_df["target_task"]]
heat = ens_df[ens_methods].values
heat_for_argmax = np.where(np.isnan(heat), -np.inf, heat)
best_col = np.argmax(heat_for_argmax, axis=1)
text_matrix = [
    [
        ("" if np.isnan(v)
         else (f"<b>{v:.2f}</b>" if j == best_col[i] else f"{v:.2f}"))
        for j, v in enumerate(row)
    ]
    for i, row in enumerate(heat)
]
fig = go.Figure(data=go.Heatmap(
    z=heat, x=ens_labels, y=short_names,
    text=text_matrix, texttemplate="%{text}", textfont=dict(size=10),
    colorscale="RdYlGn", zmin=0, zmax=1, colorbar=dict(title="accuracy"),
))
for i, j in enumerate(best_col):
    fig.add_shape(
        type="rect", xref="x", yref="y",
        x0=j - 0.5, x1=j + 0.5, y0=i - 0.5, y1=i + 0.5,
        line=dict(color="black", width=2.5),
        fillcolor="rgba(0,0,0,0)", layer="above",
    )
fig.update_layout(
    title="Ensemble heatmap (winner per row outlined)",
    width=1050, height=max(450, 24 * len(ens_df)),
    yaxis=dict(autorange="reversed"),
)
fig.show()

# 3) Summary
summary = ens_df[ens_methods].agg(["mean", "median", "std"]).T.round(3)
summary.index = ens_labels
# How often does each ensemble beat each member individually?
member_cols = ens_methods[:4]
for ens_col, ens_lab in zip(ens_methods[4:], ens_labels[4:]):
    summary.loc[ens_lab, "wins_vs_best_member"] = float(
        (ens_df[ens_col] > ens_df[member_cols].max(axis=1)).sum()
    )
summary["best_count"] = (
    ens_df[ens_methods].rank(axis=1, ascending=False, method="min") == 1
).sum().values
print("\nPer-method summary (members + ensembles):\n")
print(summary)


Completed: 23 / 27 tasks
Avg # members per ensemble: 4.00 / 4



Per-method summary (members + ensembles):

                           mean  median    std  wins_vs_best_member  \
multi-source OT           0.707   0.735  0.122                  NaN   
multi-source OT + burr    0.803   0.826  0.106                  NaN   
MS-TrAdaBoost             0.866   0.880  0.068                  NaN   
DeepJDOT                  0.809   0.838  0.103                  NaN   
ENSEMBLE — hard majority  0.861   0.873  0.060                  4.0   
ENSEMBLE — hard strict    0.832   0.838  0.093                  3.0   
ENSEMBLE — soft average   0.839   0.840  0.093                  5.0   

                          best_count  
multi-source OT                    0  
multi-source OT + burr             5  
MS-TrAdaBoost                      8  
DeepJDOT                           4  
ENSEMBLE — hard majority           6  
ENSEMBLE — hard strict             5  
ENSEMBLE — soft average            8  
